## Filtering transactions

Filtering transactions by dividing them into ones containing a negation (negative transactions) and ones not containing a negation (positive transactions). The filtering will be conducted based on previously created index table of transactions that contain a match to a negation pattern. The result will be two new databases: negative transactions and positive transactions.

In [1]:
import sys
sys.path.append('../../../common_code')

In [2]:
import sqlite3
from paths import PATH_ROOT
from db_operations.index_operations.index_operations import *
from db_operations.verb_transactions.filter_verb_transaction_tables import *
from db_operations.db_display import *

## Input parameters

In [3]:
INPUT_DIR = "../example_data"

NEG_MATCHES_DB = f"{INPUT_DIR}/negation_matches.db"
TRANSACTION_DB = f"{INPUT_DIR}/transactions.db"
NEG_TR_DB = f"{INPUT_DIR}/negative_transactions.db" # negated transactions
POS_TR_DB = f"{INPUT_DIR}/positive_transactions.db" # positive (non-negated) transactions*

## Data processing

In [4]:
con = sqlite3.connect(NEG_MATCHES_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS transactions')
cur.execute(f'ATTACH DATABASE "{NEG_TR_DB}" AS neg_tr')
cur.execute(f'ATTACH DATABASE "{POS_TR_DB}" AS pos_tr')

# negated transactions
filter_verb_transaction_tables(
    conn=con,
    source_schema='transactions',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='neg_tr',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='main',
    ids_table='neg_phrase_matches',
    ids_column='head_id',
    delete_if_exists=True,
    copy_indexes=True,
    verbose= True
)

# non-negated transactions
index_difference(cur,
                 index_tbl_1='transactions.transaction_head',
                 id_col_1='id',
                 index_tbl_2='neg_phrase_matches',
                 id_col_2='head_id',
                 output_tbl='pos_tr.index_tbl')


filter_verb_transaction_tables(
    conn=con,
    source_schema='transactions',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='pos_tr',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='pos_tr',
    ids_table='index_tbl',
    ids_column='idx',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True
)

Deleted existing table 'neg_tr.transaction_head'.
CREATE TABLE "neg_tr"."transaction_head" (id INT PRIMARY KEY, sentence_id INT, loc INT, verb TEXT, verb_compound TEXT, form TEXT, deprel TEXT, feats TEXT)
Created table 'neg_tr.transaction_head' (foreign keys ignored).
Copying indexes from 'transactions.transaction_head' to 'neg_tr.transaction_head'
Skipping auto-index or undefined index: sqlite_autoindex_transaction_head_1
Deleted existing table 'neg_tr.transaction_row'.
CREATE TABLE "neg_tr"."transaction_row" (id INT PRIMARY KEY, head_id INT, loc INT, loc_rel INT, deprel TEXT, form TEXT, lemma TEXT, feats TEXT, parent_loc INT, pos TEXT)
Created table 'neg_tr.transaction_row' (foreign keys ignored).
Copying indexes from 'transactions.transaction_row' to 'neg_tr.transaction_row'
Skipping auto-index or undefined index: sqlite_autoindex_transaction_row_1

    INSERT INTO "neg_tr"."transaction_head"
    SELECT th_source.*
    FROM "transactions"."transaction_head" AS th_source
    INNER JO

,Parameter,Schema,Table,Rows,Time (sec)
0,head_ids unique values,pos_tr,index_tbl,182,---
1,head_ids_table,pos_tr,index_tbl,182,---
2,transaction_head,transactions,transaction_head,296,---
3,transaction_row,transactions,transaction_row,810,---
4,new_transaction_head,pos_tr,transaction_head,182,0.001
5,new_transaction_row,pos_tr,transaction_row,409,0.001
6,fetching rows count,---,---,---,0.001
7,creating tables,---,---,---,0.011
8,committing result to db,---,---,---,0.003
9,total time,---,---,---,0.018


## Result

In [5]:
display_sqlite_as_dataframe(NEG_TR_DB, 'transaction_head', 10)

,id,sentence_id,loc,verb,verb_compound,form,deprel,feats
0,3,5,11,saama,pihta,saanud,root,"aux,partic,past,ps"
1,54,40,5,saama,,saa,root,"aux,indic,neg,pres,ps"
2,97,63,11,võtma,,võta,acl:relcl,"aux,indic,neg,pres,ps"
3,148,88,5,tundma,,tunne,advcl,"imper,mod,neg,pres,ps,ps2,sg"
4,185,107,6,mõtlema,,mõelnud,root,"aux,impf,indic,neg,ps"
5,212,124,3,häirima,,häiri,root,"indic,main,neg,pres,ps"
6,286,161,13,minema,,läinud,acl:relcl,"aux,partic,past,ps"
7,358,202,6,meeldima,,meeldi,advcl,"imper,main,neg,pres,ps,ps2,sg"
8,365,204,16,klappima,,klapi,ccomp,"indic,main,neg,pres,ps"
9,375,208,9,meeldima,,meeldi,advcl,"imper,main,neg,pres,ps,ps2,sg"


In [6]:
display_sqlite_as_dataframe(NEG_TR_DB, 'transaction_row', 10)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos
0,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S
1,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S
2,6,3,10,-1,aux,ei,ei,"aux,neg",None,V
3,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S
4,8,3,13,2,compound:prt,pihta,pihta,,None,D
5,89,54,3,-2,xcomp,rikkaks,rikas,"pos,sg,tr",None,A
6,90,54,4,-1,aux,ei,ei,"aux,neg",None,V
7,149,97,7,-4,nsubj,kes,kes,"nom,sg",None,P
8,150,97,8,-3,advmod,üldse,üldse,,None,D
9,151,97,9,-2,obj,napsu,naps,"com,part,sg",None,S


In [7]:
display_sqlite_as_dataframe(POS_TR_DB, 'index_tbl', 10)

,idx
0,6
1,10
2,17
3,23
4,28
5,31
6,48
7,49
8,53
9,56


In [8]:
display_sqlite_as_dataframe(POS_TR_DB, 'transaction_head', 10)

,id,sentence_id,loc,verb,verb_compound,form,deprel,feats
0,6,6,2,kulmineeruma,,kulmineerus,root,"af,impf,indic,main,ps,ps3,sg"
1,10,10,3,tulema,,tuli,root,"af,aux,impf,indic,ps,ps3,sg"
2,17,16,1,hakkama,,Hakkasin,root,"af,impf,indic,mod,ps,ps1,sg"
3,23,20,1,alustama,,Alustasid,root,"af,impf,indic,mod,pl,ps,ps3"
4,28,23,2,hakkama,,hakkasid,root,"af,impf,indic,main,pl,ps,ps3"
5,31,25,1,tegutsema,,Tegutsesin,root,"af,aux,impf,indic,ps,ps1,sg"
6,48,36,13,minema,,läks,conj,"af,impf,indic,mod,ps,ps3,sg"
7,49,36,17,tegelema,,tegelesin,conj,"af,impf,indic,main,ps,ps1,sg"
8,53,39,5,tulema,,tuled,root,"af,indic,mod,pres,ps,ps2,sg"
9,56,41,2,meeldima,,meeldib,root,"af,indic,main,pres,ps,ps3,sg"


In [9]:
display_sqlite_as_dataframe(POS_TR_DB, 'transaction_row', 10)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos
0,14,6,1,-1,nsubj,Esinemine,esinemine,"com,nom,sg",None,S
1,15,6,4,1,obl,purukspeksmisega,purukspeksmine,"com,kom,sg",None,S
2,18,10,2,-1,advmod,sageli,sageli,,None,D
3,19,10,4,1,obl,sul,sina,"ad,sg",None,P
4,20,10,9,2,csubj,anda,andma,"inf,main",None,V
5,28,17,6,1,xcomp,tegema,tegema,"ill,mod,ps,sup",None,V
6,29,17,11,2,advcl,viie-,viis,"card,gen,l,sg",None,N
7,36,23,3,1,obl,muusikaga,muusika,"com,kom,sg",None,S
8,45,28,1,-1,advmod,Tasapisi,tasapisi,,None,D
9,46,28,3,1,advmod,aga,aga,"crd,sub",None,J


NB! Andmebaasi *positive_transactions.db* tabelisse *transaction_head* on mõnel juhul jäänud alles verbivormid, mille *feats* väärtuses esineb eituse märge. Need juhud on märgitud eituseks eksikombel, tegemist on valdavalt *nud/tud-* kesksõnadega (nt *jäänud*, *arvanud* jne), tingiva kõneviisi vormidega (*tuleks*, *peaks* jne) või käskiva kõneviisi vormidega (*vabandage*). Teatud konstruktsioonides võivad need vormid tõepoolest olla osa eituskonstruktsioonist, millest tõenäoliselt on ka viga *feats* väärtuses tulenenud (nt 'pole *teinud*' vs 'on *teinud*', '*tuleks* teha' vs 'ei *tuleks* teha', '*vabandage*' vs 'ärge *vabandage*'). Tabelisse *transaction_row* on jäänud alles eitusvormis sõnad, mis ei kuulu fraasi peaverbi (*transaction_head* tabelis oleva verbi) juurde.


Andmebaasi *negative_transactions.db* tabelis *transaction_head* esineb kohati verbe, mille *feats* väärtuses puudub eituse märge. Tegemist on siiski eitusvormis verbidega, mille juurde kuuluv eitussõna (*ei*, *ära*, *pole* jne) on *transaction_row* tabelis.